# EN3160 Assignment 3 – Neural Networks
**Name:** Maldeniya P. A. D. G. R. (Ransara Maldeniya)  
**Index No:** 230390A  
**GitHub:**  https://github.com/ransara3/EN3160-Assignment-3-Neural-Networks.git 

Contents: Q1 manual two-layer dense network (CIFAR-10) · Q2 LeNet-5 vs modern CNN (MNIST) · Q3 ResNet18 transfer learning (hymenoptera)

> Run all cells top to bottom. Text marked **[Fill in]** must be completed with your own numbers after running.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np, time, copy, os, zipfile, urllib.request

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# Question 1 – Two-layer dense network with manual backprop (CIFAR-10)

**Changes to Listing 1**
- **(a)** Added a hidden layer of 100 nodes with sigmoid activation: `x → [W1,b1] → sigmoid → [W2,b2] → logits`.
- **(b)** Replaced MSE with softmax cross-entropy: $L=-\frac1N\sum_i \log p_{i,y_i}$, with $p=\text{softmax}(z)$.
- **(c)** Train for 10 epochs; report train and test accuracy.

**Backprop derivation** (N = batch size, $h=\sigma(z_1)$, $z_2=hW_2+b_2$):
- $\partial L/\partial z_2 = (p - y_{onehot})/N$
- $\partial L/\partial W_2 = h^T\,\partial L/\partial z_2$, $\;\partial L/\partial b_2=\sum \partial L/\partial z_2$
- $\partial L/\partial h = \partial L/\partial z_2\,W_2^T$
- $\partial L/\partial z_1 = \partial L/\partial h \odot h(1-h)$ (sigmoid derivative)
- $\partial L/\partial W_1 = x^T\,\partial L/\partial z_1$, $\;\partial L/\partial b_1=\sum \partial L/\partial z_1$

The softmax + cross-entropy gradient simplifies to $p - y$, which is why no explicit softmax Jacobian is needed.

In [ ]:
# 1. Data loading (same as Listing 1)
transform = transforms.Compose([transforms.ToTensor(),
                                transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))])
batch_size = 50
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)
classes = ('plane','car','bird','cat','deer','dog','frog','horse','ship','truck')

In [ ]:
# 2. Parameters
Din, H, K = 3*32*32, 100, 10
std = 1e-2
w1 = torch.randn(Din, H) * std;  b1 = torch.zeros(H)
w2 = torch.randn(H, K) * std;    b2 = torch.zeros(K)

epochs   = 10
lr       = 0.05
lr_decay = 0.9
reg      = 0.0

def forward(x):
    h = torch.sigmoid(x.mm(w1) + b1)
    return h, h.mm(w2) + b2

def cross_entropy(logits, labels):
    return -torch.log_softmax(logits, dim=1)[torch.arange(len(labels)), labels].mean()

def evaluate(loader):
    correct = total = 0; loss_sum = 0.0
    with torch.no_grad():
        for inputs, labels in loader:
            x = inputs.view(inputs.shape[0], -1)
            _, logits = forward(x)
            loss_sum += cross_entropy(logits, labels).item() * len(labels)
            correct += (logits.argmax(1) == labels).sum().item(); total += len(labels)
    return loss_sum/total, 100*correct/total

In [ ]:
# 3. Training loop (manual backprop)
loss_history, ep_train_loss, ep_train_acc, ep_test_acc = [], [], [], []
for t in range(epochs):
    running = 0.0
    for inputs, labels in trainloader:
        N = inputs.shape[0]
        x = inputs.view(N, -1)
        y1h = nn.functional.one_hot(labels, K).float()

        # forward
        h, logits = forward(x)
        loss = cross_entropy(logits, labels) + reg*(w1.pow(2).sum() + w2.pow(2).sum())
        loss_history.append(loss.item()); running += loss.item()

        # backward
        p  = torch.softmax(logits, dim=1)
        dz2 = (p - y1h) / N
        dw2 = h.t().mm(dz2) + 2*reg*w2
        db2 = dz2.sum(0)
        dh  = dz2.mm(w2.t())
        dz1 = dh * h * (1 - h)
        dw1 = x.t().mm(dz1) + 2*reg*w1
        db1 = dz1.sum(0)

        # update
        w1 -= lr*dw1; b1 -= lr*db1; w2 -= lr*dw2; b2 -= lr*db2

    _, tr_acc = evaluate(trainloader); _, te_acc = evaluate(testloader)
    ep_train_loss.append(running/len(trainloader)); ep_train_acc.append(tr_acc); ep_test_acc.append(te_acc)
    print(f'Epoch {t+1}/{epochs}  loss {running/len(trainloader):.4f}  train acc {tr_acc:.2f}%  test acc {te_acc:.2f}%')
    lr *= lr_decay

In [ ]:
# 4. Plots
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(loss_history); ax[0].set_title('Loss history (per iteration)'); ax[0].set_xlabel('Iteration'); ax[0].set_ylabel('Cross-entropy loss')
ax[1].plot(range(1,epochs+1), ep_train_acc, label='Train'); ax[1].plot(range(1,epochs+1), ep_test_acc, label='Test')
ax[1].set_title('Accuracy per epoch'); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Accuracy (%)'); ax[1].legend()
plt.tight_layout(); plt.show()

_, q1_train = evaluate(trainloader); _, q1_test = evaluate(testloader)
print(f'Q1 Training accuracy: {q1_train:.2f}%')
print(f'Q1 Test accuracy:     {q1_test:.2f}%')

### Q1 Discussion  **[Fill in after running]**
- Final train accuracy: **__%**, test accuracy: **__%**.
- The loss decreases steadily; the sigmoid hidden layer adds non-linearity over the single-layer model, so accuracy beats a purely linear classifier on raw pixels.
- Cross-entropy gives larger gradients than MSE when predictions are confidently wrong, so it trains faster and suits classification (outputs interpreted as class probabilities).
- A fully connected network ignores spatial structure of images, so accuracy plateaus well below that of CNNs (roughly 50% range expected). A small train–test gap indicates little overfitting at 10 epochs.
- Sigmoid saturation and small initialisation make learning slow; that's why lr, decay and init std were tuned (lr = 0.05, decay = 0.9, std = 1e-2).